# Unit 8 Part 1: PPO (Proximal Policy Optimization) — LunarLander-v3

PPO를 **CleanRL 방식으로 PyTorch로 직접 구현**하고 `LunarLander-v3`에서 훈련합니다.

### 이전 유닛과의 차이점

| 항목 | Unit 6 (A2C/SB3) | Unit 8 (PPO/CleanRL) |
|---|---|---|
| **구현 방식** | SB3 라이브러리 사용 | **PyTorch로 직접 구현** |
| **알고리즘** | A2C | **PPO** |
| **코드 구조** | 래퍼 API | 단일 파일 (`ppo.py`) |

### PPO 핵심 아이디어
```
기존 Policy Gradient: 업데이트가 너무 크면 학습 불안정
PPO: Clipped Surrogate Objective로 업데이트 크기를 제한

Loss = -min(
    ratio × A,                          ← 일반 PG
    clip(ratio, 1-ε, 1+ε) × A          ← 클리핑된 버전
)
ratio = π_new(a|s) / π_old(a|s)
```

### CleanRL이란?
각 알고리즘을 **단일 파일**로 구현한 RL 라이브러리입니다.  
코드를 직접 읽고 이해하기 쉬워서 학습용으로 적합합니다.

---
## 목차
1. 기존 충돌 패키지 정리 및 환경 설치
2. Google Drive 마운트
3. 가상 디스플레이 설정
4. HF 유틸리티 함수 정의
5. PPO 전체 코드 (`ppo.py`) 생성
6. HF 로그인
7. 훈련 실행
8. 훈련 결과 확인 (TensorBoard)
9. 훈련 영상 Drive에서 확인


---
## 1. 기존 충돌 패키지 정리 및 환경 설치
이전 실행에서 남아있는 구버전 `gym` 패키지와 충돌을 방지하기 위해 완전히 삭제 후 최신 `gymnasium` 기반으로 재설치합니다.

In [ ]:
%%capture
# 꼬여있는 기존 gym 완전 삭제
!pip uninstall -y gym

# 시스템 패키지 설치
!apt update && apt install -y python3-opengl ffmpeg xvfb swig cmake build-essential
!pip install pyvirtualdisplay

# 최신 gymnasium 및 Box2D 설치 (v3 지원)
!pip install 'gymnasium[box2d]'
!pip install imageio imageio-ffmpeg
!pip install huggingface_hub wasabi

---
## 2. Google Drive 마운트

훈련 결과(TensorBoard 로그, 영상)를 Drive에 저장합니다.

```
Google Drive/RL_Course/Unit8_PPO/
├── runs/          ← TensorBoard 로그
└── videos/        ← 훈련 중 캡처된 영상
```


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ✏️ 저장 경로를 원하는 대로 변경하세요.
DRIVE_BASE  = '/content/drive/MyDrive/RL_Course/Unit8_PPO'
DRIVE_RUNS  = f'{DRIVE_BASE}/runs'
DRIVE_VIDEOS = f'{DRIVE_BASE}/videos'

os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(DRIVE_VIDEOS, exist_ok=True)

os.environ['DRIVE_BASE']   = DRIVE_BASE
os.environ['DRIVE_RUNS']   = DRIVE_RUNS
os.environ['DRIVE_VIDEOS'] = DRIVE_VIDEOS

print('✅ Drive 마운트 완료!')
print(f'   로그  : {DRIVE_RUNS}')
print(f'   영상  : {DRIVE_VIDEOS}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive 마운트 완료!
   로그  : /content/drive/MyDrive/RL_Course/Unit8_PPO/runs
   영상  : /content/drive/MyDrive/RL_Course/Unit8_PPO/videos


---
## 3. 가상 디스플레이 설정


In [3]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print('✅ 가상 디스플레이 시작')

✅ 가상 디스플레이 시작


---
## 4. HF 유틸리티 함수 정의


---
## 5. PPO 전체 코드 (`ppo.py`) 생성

LunarLander-v3 환경에 맞춰 `ppo.py`를 재생성합니다. 기존 파일이 있다면 덮어씌웁니다.

In [15]:
import os, subprocess, shutil

# 기존 ppo.py 파일이 문제를 일으킬 수 있으므로 명시적 삭제
if os.path.exists('/content/ppo.py'):
    os.remove('/content/ppo.py')

DRIVE_BASE = os.environ.get('DRIVE_BASE', '/content/drive/MyDrive/RL_Course/Unit8_PPO')

# 여러 줄의 문자열을 안전하게 작성하기 위해 멀티라인 문자열(""") 사용
code = """import argparse, os, random, time, shutil, json, datetime, tempfile
from distutils.util import strtobool
from pathlib import Path
import gymnasium as gym  # 무조건 gymnasium 사용
import numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter
import imageio
from huggingface_hub import HfApi, upload_folder
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from wasabi import Printer
msg = Printer()

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--exp-name", type=str, default=os.path.basename(__file__).rstrip(".py"))
    parser.add_argument("--seed", type=int, default=1)
    parser.add_argument("--torch-deterministic", type=lambda x: bool(strtobool(x)), default=True, nargs="?")
    parser.add_argument("--cuda", type=lambda x: bool(strtobool(x)), default=True, nargs="?")
    parser.add_argument("--capture-video", type=lambda x: bool(strtobool(x)), default=False, nargs="?")
    parser.add_argument("--env-id", type=str, default="LunarLander-v3")
    parser.add_argument("--total-timesteps", type=int, default=500000)
    parser.add_argument("--learning-rate", type=float, default=2.5e-4)
    parser.add_argument("--num-envs", type=int, default=4)
    parser.add_argument("--num-steps", type=int, default=128)
    parser.add_argument("--anneal-lr", type=lambda x: bool(strtobool(x)), default=True, nargs="?")
    parser.add_argument("--gae", type=lambda x: bool(strtobool(x)), default=True, nargs="?")
    parser.add_argument("--gamma", type=float, default=0.99)
    parser.add_argument("--gae-lambda", type=float, default=0.95)
    parser.add_argument("--num-minibatches", type=int, default=4)
    parser.add_argument("--update-epochs", type=int, default=4)
    parser.add_argument("--norm-adv", type=lambda x: bool(strtobool(x)), default=True, nargs="?")
    parser.add_argument("--clip-coef", type=float, default=0.2)
    parser.add_argument("--clip-vloss", type=lambda x: bool(strtobool(x)), default=True, nargs="?")
    parser.add_argument("--ent-coef", type=float, default=0.01)
    parser.add_argument("--vf-coef", type=float, default=0.5)
    parser.add_argument("--max-grad-norm", type=float, default=0.5)
    parser.add_argument("--target-kl", type=float, default=None)
    parser.add_argument("--repo-id", type=str, default="YOUR_HF_USERNAME/ppo-LunarLander-v3")
    parser.add_argument("--drive-base", type=str, default=os.environ.get("DRIVE_BASE", "/content"))
    args = parser.parse_args()
    args.batch_size = int(args.num_envs * args.num_steps)
    args.minibatch_size = int(args.batch_size // args.num_minibatches)
    return args

def make_env(env_id, seed, idx, capture_video, run_name):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        if capture_video and idx == 0:
            env = gym.wrappers.RecordVideo(env, f"videos/{run_name}")
        return env
    return thunk

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, envs):
        super().__init__()
        obs_dim = np.array(envs.single_observation_space.shape).prod()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )
        self.actor = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, envs.single_action_space.n), std=0.01),
        )
    def get_value(self, x): return self.critic(x)
    def get_action_and_value(self, x, action=None):
        logits = self.actor(x); probs = Categorical(logits=logits)
        if action is None: action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

def _evaluate_agent(env, n_eval_episodes, policy):
    episode_rewards = []
    for _ in range(n_eval_episodes):
        state, _ = env.reset()
        done = False; total = 0.0
        while not done:
            state = torch.as_tensor(state, dtype=torch.float32, device=device)
            action, _, _, _ = policy.get_action_and_value(state)
            state, reward, terminated, truncated, _ = env.step(action.cpu().numpy())
            done = terminated or truncated
            total += reward
        episode_rewards.append(total)
    return np.mean(episode_rewards), np.std(episode_rewards)

def record_video(env, policy, out_directory, fps=30):
    images = []
    state, _ = env.reset()
    images.append(env.render())
    done = False
    while not done:
        state = torch.as_tensor(state, dtype=torch.float32, device=device)
        action, _, _, _ = policy.get_action_and_value(state)
        state, _, terminated, truncated, _ = env.step(action.cpu().numpy())
        done = terminated or truncated
        images.append(env.render())
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)

def generate_metadata(model_name, env_id, mean_reward, std_reward):
    metadata = {"tags": [env_id, "ppo", "deep-reinforcement-learning",
                         "reinforcement-learning", "custom-implementation", "deep-rl-course"]}
    eval_meta = metadata_eval_result(
        model_pretty_name=model_name, task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning", metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_id, dataset_id=env_id,
    )
    return {**metadata, **eval_meta}

def package_to_hub(repo_id, model, hyperparameters, eval_env,
                   video_fps=30, commit_message="Push agent to the Hub",
                   token=None, logs=None, drive_base=None):
    msg.info("에이전트 평가, 영상 생성, HF Hub 업로드를 시작합니다...")
    repo_url = HfApi().create_repo(repo_id=repo_id, token=token, private=False, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmpdirname:
        tmpdirname = Path(tmpdirname)
        torch.save(model.state_dict(), tmpdirname / "model.pt")
        mean_reward, std_reward = _evaluate_agent(eval_env, 10, model)
        with open(tmpdirname / "results.json", "w") as f:
            json.dump({"env_id": hyperparameters.env_id,
                       "mean_reward": mean_reward, "std_reward": std_reward,
                       "n_evaluation_episodes": 10,
                       "eval_datetime": datetime.datetime.now().isoformat()}, f)
        video_env = gym.make(hyperparameters.env_id, render_mode="rgb_array")
        video_path = tmpdirname / "replay.mp4"
        record_video(video_env, model, video_path, video_fps)
        video_env.close()
        if drive_base and os.path.exists(drive_base):
            drive_video_dir = os.path.join(drive_base, "videos")
            os.makedirs(drive_video_dir, exist_ok=True)
            shutil.copy(video_path, os.path.join(drive_video_dir,
                f"replay_{hyperparameters.env_id}_{int(time.time())}.mp4"))
            print("영상 Drive 저장 완료")
        metadata = generate_metadata("PPO", hyperparameters.env_id, mean_reward, std_reward)
        readme_path = tmpdirname / "README.md"
        readme_path.write_text(
            "# PPO Agent Playing " + hyperparameters.env_id + "\\n\\n"
            + "평균 보상: " + f"{mean_reward:.2f} +/- {std_reward:.2f}" + "\\n",
            encoding="utf-8")
        metadata_save(readme_path, metadata)
        if logs and Path(logs).exists():
            repo_logdir = tmpdirname / "logs"
            if repo_logdir.exists(): shutil.rmtree(repo_logdir)
            shutil.copytree(logs, repo_logdir)
            if drive_base and os.path.exists(drive_base):
                drive_log = os.path.join(drive_base, "runs", os.path.basename(logs))
                if not os.path.exists(drive_log): shutil.copytree(logs, drive_log)
        repo_url = upload_folder(repo_id=repo_id, folder_path=tmpdirname,
            path_in_repo="", commit_message=commit_message, token=token)
        msg.info(f"업로드 완료: {repo_url}")
        print(f"평균 보상: {mean_reward:.2f} +/- {std_reward:.2f}")
    return repo_url

if __name__ == "__main__":
    args = parse_args()
    run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"
    writer = SummaryWriter(f"runs/{run_name}")
    writer.add_text("hyperparameters",
        "|param|value|\\n|-|-|\\n" + "\\n".join([f"|{k}|{v}|" for k, v in vars(args).items()]))
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    torch.backends.cudnn.deterministic = args.torch_deterministic
    device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")
    print(f"사용 디바이스: {device}")
    envs = gym.vector.SyncVectorEnv(
        [make_env(args.env_id, args.seed+i, i, args.capture_video, run_name) for i in range(args.num_envs)]
    )
    assert isinstance(envs.single_action_space, gym.spaces.Discrete)
    agent = Agent(envs).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)
    obs      = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape, dtype=torch.float32).to(device)
    actions  = torch.zeros((args.num_steps, args.num_envs), dtype=torch.int64).to(device)
    logprobs = torch.zeros((args.num_steps, args.num_envs), dtype=torch.float32).to(device)
    rewards  = torch.zeros((args.num_steps, args.num_envs), dtype=torch.float32).to(device)
    dones    = torch.zeros((args.num_steps, args.num_envs), dtype=torch.float32).to(device)
    values   = torch.zeros((args.num_steps, args.num_envs), dtype=torch.float32).to(device)
    global_step = 0; start_time = time.time()
    next_obs, _ = envs.reset(seed=args.seed)
    next_obs  = torch.as_tensor(next_obs, dtype=torch.float32, device=device)
    next_done = torch.zeros(args.num_envs, dtype=torch.float32).to(device)
    num_updates = args.total_timesteps // args.batch_size
    for update in range(1, num_updates + 1):
        if args.anneal_lr:
            frac = 1.0 - (update - 1.0) / num_updates
            optimizer.param_groups[0]["lr"] = frac * args.learning_rate
        for step in range(args.num_steps):
            global_step += args.num_envs
            obs[step] = next_obs; dones[step] = next_done
            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(next_obs)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob
            next_obs, reward, terminated, truncated, info = envs.step(action.cpu().numpy())
            done = np.logical_or(terminated, truncated)
            rewards[step] = torch.as_tensor(reward, dtype=torch.float32, device=device).view(-1)
            next_obs  = torch.as_tensor(next_obs, dtype=torch.float32, device=device)
            next_done = torch.as_tensor(done, dtype=torch.float32, device=device)
            if "final_info" in info:
                for item in info["final_info"]:
                    if item and "episode" in item:
                        print(f"global_step={global_step}, episodic_return={item['episode']['r']:.2f}")
                        writer.add_scalar("charts/episodic_return", item["episode"]["r"], global_step)
                        writer.add_scalar("charts/episodic_length", item["episode"]["l"], global_step)
                        break
        with torch.no_grad():
            next_value = agent.get_value(next_obs).reshape(1, -1)
            advantages = torch.zeros_like(rewards).to(device)
            lastgaelam = 0
            for t in reversed(range(args.num_steps)):
                nextnonterminal = 1.0 - (next_done if t == args.num_steps-1 else dones[t+1])
                nextvalues = next_value if t == args.num_steps-1 else values[t+1]
                delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
                advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
            returns = advantages + values
        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape(-1)
        b_advantages = advantages.reshape(-1); b_returns = returns.reshape(-1); b_values = values.reshape(-1)
        b_inds = np.arange(args.batch_size); clipfracs = []
        for epoch in range(args.update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, args.batch_size, args.minibatch_size):
                mb_inds = b_inds[start:start + args.minibatch_size]
                _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], b_actions.long()[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]; ratio = logratio.exp()
                with torch.no_grad():
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfracs += [((ratio - 1.0).abs() > args.clip_coef).float().mean().item()]
                mb_advantages = b_advantages[mb_inds]
                if args.norm_adv:
                    mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                newvalue = newvalue.view(-1)
                if args.clip_vloss:
                    v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                    v_clipped = b_values[mb_inds] + torch.clamp(newvalue - b_values[mb_inds], -args.clip_coef, args.clip_coef)
                    v_loss = 0.5 * torch.max(v_loss_unclipped, (v_clipped - b_returns[mb_inds])**2).mean()
                else:
                    v_loss = 0.5 * ((newvalue - b_returns[mb_inds])**2).mean()
                entropy_loss = entropy.mean()
                loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
                optimizer.step()
            if args.target_kl is not None and approx_kl > args.target_kl: break
        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y
        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
        writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
        writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
        writer.add_scalar("losses/explained_variance", explained_var, global_step)
        print(f"SPS: {int(global_step / (time.time() - start_time))}")
    envs.close(); writer.close()
    eval_env = gym.make(args.env_id)
    package_to_hub(repo_id=args.repo_id, model=agent, hyperparameters=args,
                   eval_env=eval_env, logs=f"runs/{run_name}",
                   drive_base=args.drive_base)
"""

with open('/content/ppo.py', 'w') as f:
    f.write(code)

result = subprocess.run(['python', '-m', 'py_compile', '/content/ppo.py'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ ppo.py 생성 및 문법 검사 완료 (gymnasium 기반)')
else:
    print('❌ 문법 오류:'); print(result.stderr)


✅ ppo.py 생성 및 문법 검사 완료 (gymnasium 기반)


---
## 6. HF 로그인

1. [HF 계정 생성](https://huggingface.co/join)
2. [Write 토큰 발급](https://huggingface.co/settings/tokens)


In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰 입력
login(token='XXXXXXXXXX')
!git config --global credential.helper store

---
## 7. 훈련 실행

`ppo.py`를 실행하여 v3 환경에서 PPO 훈련을 시작합니다.

| 파라미터 | 값 | 설명 |
|---|---|---|
| `--env-id` | `LunarLander-v3` | 훈련 환경 |
| `--total-timesteps` | 50,000 | 총 스텝 수 (더 좋은 결과를 원하면 500,000~1,000,000 권장) |
| `--repo-id` | `username/ppo-LunarLander-v3` | HF Hub 저장소 |

> ⏱️ 50,000 스텝 기준 GPU: **약 2~3분**  
> 더 좋은 성능을 위해 `--total-timesteps=500000` 이상을 권장합니다.


In [33]:
import os

DRIVE_BASE = os.environ.get('DRIVE_BASE', '/content/drive/MyDrive/RL_Course/Unit8_PPO')

# ✏️ repo-id를 본인의 HF 사용자명으로 변경하세요.
!python /content/ppo.py \
    --env-id="LunarLander-v3" \
    --repo-id="DitDahDitDit/ppo-LunarLander-v3_NEW_HANDS_ON" \
    --total-timesteps=500000 \
    --drive-base="{DRIVE_BASE}"

2026-08-05 10:06:18.028373: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
사용 디바이스: cuda
SPS: 606
SPS: 872
SPS: 1017
SPS: 1113
SPS: 1177
SPS: 1229
SPS: 1270
SPS: 1298
SPS: 1326
SPS: 1351
SPS: 1368
SPS: 1382
SPS: 1395
SPS: 1401
SPS: 1412
SPS: 1419
SPS: 1430
SPS: 1436
SPS: 1431
SPS: 1415
SPS: 1408
SPS: 1398
SPS: 1384
SPS: 1373
SPS: 1368
SPS: 1366
SPS: 1364
SPS: 1355
SPS: 1347
SPS: 1354
SPS: 1361
SPS: 1369
SPS: 1377
SPS: 1381
SPS: 1387
SPS: 1392
SPS: 1395
SPS: 1401
SPS: 1406
SPS: 1410
SPS: 1414
SPS: 1418
SPS: 1422
SPS: 1422
SPS: 1426
SPS: 1429
SPS: 1433
SPS: 1435
SPS: 1437
SPS: 1438
SPS: 1441
SPS: 1442
SPS: 1444
SPS: 1447
SPS: 1448
SPS: 1451
SPS: 1453
SPS: 1456
SPS: 1459
SPS: 1461
SPS: 1456
SPS: 1451
SPS: 1446
SPS: 1443
SPS: 1435
SPS: 1430
SPS: 142

---
## 8. 훈련 결과 확인 (TensorBoard)


In [36]:
%load_ext tensorboard
%tensorboard --logdir /content/runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

---
## 9. 훈련 영상 Drive에서 확인


In [37]:
import glob, os
from IPython.display import Video, display

DRIVE_VIDEOS = os.environ.get('DRIVE_VIDEOS',
    '/content/drive/MyDrive/RL_Course/Unit8_PPO/videos')

videos = sorted(glob.glob(f'{DRIVE_VIDEOS}/*.mp4'))

if videos:
    print(f'총 {len(videos)}개 영상:\n')
    for v in videos:
        print(f'  {v}')
    print('\n최신 영상 재생:')
    display(Video(videos[-1], embed=True, width=500))
else:
    print('저장된 영상이 없습니다.')
    print('훈련 셀을 먼저 실행하세요.')

총 2개 영상:

  /content/drive/MyDrive/RL_Course/Unit8_PPO/videos/replay_LunarLander-v3_1785923398.mp4
  /content/drive/MyDrive/RL_Course/Unit8_PPO/videos/replay_LunarLander-v3_1785924754.mp4

최신 영상 재생:
